In [ ]:
# PROJECT IDEA AND GROUP FINDER PORTAL
USER_FILE = "users.txt"
PROJECT_FILE = "projects.txt"
def load_users():
    """Reads users from users.txt into a list of dictionaries."""
    users = []
    try:
        with open(USER_FILE, "r") as user_file:
            for line in user_file:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("|")
                if len(parts) == 4:
                    roll, name, password, skills_str = parts
                    skills = ([s.strip() for s in skills_str.split(",")]if skills_str else []
                    )
                    users.append({"roll_number": roll,"name": name,"password": password,"skills": skills})
    except FileNotFoundError:
        pass                   
    except Exception as e:
        print(f"[Warning] Error loading users: {e}")
    return users
def save_users(users):
    """Saves current users list to users.txt."""
    try:
        with open(USER_FILE, "w") as user_file:
            for u in users:
                skills_str = ",".join(u["skills"])
                user_file.write(f"{u['roll_number']}|{u['name']}|{u['password']}|{skills_str}\n")
        return True
    except Exception as e:
        print(f"[Error] Error saving users: {e}")
        return False
def load_projects():
    """Reads projects from projects.txt into a list of dictionaries."""
    projects = []
    try:
        with open(PROJECT_FILE, "r") as project_file:
            for line in project_file:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("|")
                if len(parts) == 7:
                    p_id, title, desc, owner, req_skills, max_m, members_str = (parts)
                    projects.append({"project_id": p_id,"title": title,"description": desc,"owner": owner,"required_skills": [s.strip() for s in req_skills.split(",") if s.strip()],"max_members": int(max_m),"members": [m.strip() for m in members_str.split(",") if m.strip()]})
    except FileNotFoundError:
        pass  
    except Exception as e:
        print(f"[Warning] Error loading projects: {e}")
    return projects
def save_projects(projects):
    """Saves current projects list to projects.txt."""
    try:
        with open(PROJECT_FILE, "w") as project_file:
            for p in projects:
                req_skills = ",".join(p["required_skills"])
                members = ",".join(p["members"])
                project_file.write(f"{p['project_id']}|{p['title']}|{p['description']}|{p['owner']}|{req_skills}|{p['max_members']}|{members}\n")
        return True
    except Exception as e:
        print(f"[Error] Error saving projects: {e}")
        return False
# REQUIRED PORTAL FUNCTIONS
def register_user(users):
    """Registers a new student account."""
    print("\n Register Student Account ")
    roll = input("Enter Roll Number (e.g. 25L-5150): ").strip()
    for u in users:
        if u["roll_number"].lower() == roll.lower():
            print("❌ Error: A student with this Roll Number already exists!")
            return users, None
    name = input("Enter Full Name: ").strip()
    password = input("Enter Password: ").strip()
    skills_input = input("Enter Skills (comma-separated): ")
    skills = [s.strip() for s in skills_input.split(",") if s.strip()]
    new_user = {"roll_number": roll,"name": name,"password": password,"skills": skills}
    users.append(new_user)
    if save_users(users):
        print(f"✅ Account registered successfully for {name}!")
        return users, new_user
    return users, None
def user_login(users):
    """Logs in an existing student."""
    print("\n Student Login ")
    roll = input("Enter Roll Number: ").strip()
    password = input("Enter Password: ").strip()
    for u in users:
        if u["roll_number"].lower() == roll.lower() and u["password"] == password:
            print(f"\n🎉 Login successful! Welcome {u['name']}.")
            return u
    print("❌ Invalid Roll Number or Password.")
    return None
def add_record(projects, current_user):
    """Posts a new project idea."""
    print("\n Post New Project Idea ")
    title = input("Enter Project Title: ").strip()
    description = input("Enter Project Description: ").strip()
    skills_input = input("Required Skills (comma-separated): ")
    req_skills = [s.strip() for s in skills_input.split(",") if s.strip()]
    while True:
        try:
            max_members = int(input("Enter Team Limit (e.g. 3 or 4): ").strip())
            if max_members > 0:
                break
            print("Please enter a positive number.")
        except ValueError:
            print("❌ Invalid input! Enter a valid integer.")
    p_id = f"PRJ-{len(projects) + 1:03d}"
    new_project = {"project_id": p_id,"title": title,"description": description,"owner": current_user["roll_number"],"required_skills": req_skills,"max_members": max_members,"members": [current_user["roll_number"]]}
    projects.append(new_project)
    if save_projects(projects):
        print(f"✅ Project '{title}' created with ID {p_id}!")
    return projects
def search_record(projects):
    """Searches project records by keyword."""
    print("\n Search Projects ")
    query = input("Enter keyword to search: ").strip().lower()
    found = []
    for p in projects:
        in_title = query in p["title"].lower()
        in_desc = query in p["description"].lower()
        in_skills = any(query in s.lower() for s in p["required_skills"])
        if in_title or in_desc or in_skills:
            found.append(p)
    if not found:
        print("🔍 No matching projects found.")
        return
    print(f"\nFound {len(found)} project(s):")
    for p in found:
        print(f"\nID: {p['project_id']} | Title: {p['title']} | Owner: {p['owner']}")
        print(f"Desc: {p['description']}")
        print(f"Skills: {', '.join(p['required_skills'])}")
        print(f"Members: {len(p['members'])}/{p['max_members']} [{', '.join(p['members'])}]")
        print("-" * 40)
def join_project(projects, current_user):
    """Joins a student to an open project team."""
    print("\n Join a Project Team ")
    p_id = input("Enter Project ID (e.g. PRJ-001): ").strip().upper()
    for p in projects:
        if p["project_id"].upper() == p_id:
            if current_user["roll_number"] in p["members"]:
                print("⚠️ You are already in this group!")
                return projects
            if len(p["members"]) >= p["max_members"]:
                print("❌ Team is already full!")
                return projects
            p["members"].append(current_user["roll_number"])
            save_projects(projects)
            print(f"🎉 Joined team for '{p['title']}' successfully!")
            return projects
    print("❌ Project ID not found.")
    return projects
def generate_report(projects, users):
    """Generates portal statistics report."""
    print("\n-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-")
    print("      PORTAL SUMMARY & ANALYTICS REPORT    ")
    print("-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-")
    print(f"Total Registered Students: {len(users)}")
    print(f"Total Projects Posted: {len(projects)}")
    open_teams = sum(1 for p in projects if len(p["members"]) < p["max_members"])
    full_teams = len(projects) - open_teams
    print(f"Open Teams Available: {open_teams}")
    print(f"Completed/Full Teams: {full_teams}")
    print("-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-")
    if projects:
        print("\n ALL PROJECT DETAILS ")
        for p in projects:
            print(f"📌 [{p['project_id']}] {p['title']} (Owner: {p['owner']})")
            print(f"Required Skills: {', '.join(p['required_skills'])}")
            print(f"Team Members: {len(p['members'])}/{p['max_members']} -> {', '.join(p['members'])}")
            print("-" * 40)
# MAIN INTERACTIVE MENU LOOP
def main():
    users = load_users()
    projects = load_projects()
    current_user = None
    while True:
        print("\n-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-")
        print("  PROJECT IDEA AND GROUP FINDER PORTAL   ")
        print("-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-")
        if not current_user:
            print("1. Register Account")
            print("2. Login")
            print("3. View Summary Report (Guest)")
            print("4. Exit")
            choice = input("\nSelect option (1-4): ").strip()
            if choice == "1":
                users, current_user = register_user(users)
            elif choice == "2":
                current_user = user_login(users)
            elif choice == "3":
                generate_report(projects, users)
            elif choice == "4":
                print("\nExiting portal. All records saved. Goodbye👋😊!")
                break
            else:
                print("❌ Invalid selection! Choose 1, 2, 3, or 4.")
        else:
            print(f"Logged in: {current_user['name']} ({current_user['roll_number']})")
            print("1. Post a New Project Idea")
            print("2. Search Projects & Skills")
            print("3. Join a Project Team")
            print("4. View Summary Report")
            print("5. Logout")
            print("6. Exit")
            choice = input("\nSelect option (1-6): ").strip()
            if choice == "1":
                projects = add_record(projects, current_user)
            elif choice == "2":
                search_record(projects)
            elif choice == "3":
                projects = join_project(projects, current_user)
            elif choice == "4":
                generate_report(projects, users)
            elif choice == "5":
                print(f"\nLogged out {current_user['name']}.")
                current_user = None
            elif choice == "6":
                print("\nExiting portal. All records saved. Goodbye!")
                break
            else:
                print("❌ Invalid selection! Choose between 1 and 6.")
# Run application
if __name__ == "__main__":
    main()


-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-
  PROJECT IDEA AND GROUP FINDER PORTAL   
-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-
1. Register Account
2. Login
3. View Summary Report (Guest)
4. Exit



Select option (1-4):  2



 Student Login 


Enter Roll Number:  25l-5205
Enter Password:  11



🎉 Login successful! Welcome eshal.

-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-
  PROJECT IDEA AND GROUP FINDER PORTAL   
-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-
Logged in: eshal (25l-5205)
1. Post a New Project Idea
2. Search Projects & Skills
3. Join a Project Team
4. View Summary Report
5. Logout
6. Exit



Select option (1-6):  2



 Search Projects 


In [2]:
with open("users.txt", "r") as user_file:
    print(user_file.read())

25l-5205|eshal|11|python
25l-5555|jcnscn|12|ncdjcj

